# Validação independente FRTB — posições, elegibilidade e fluxos

**Escopo:** extração da `Base_Posicoes_Completa` para o conglomerado, `IDs_Exposicao_Sens`, `Base_Posicoes` e fluxos. Reproduz controles observados nos programas SAS apresentados; **não** equivale a validação integral do FRTB.

**Como usar:** execute a célula de inicialização abaixo uma vez e, depois, apenas o teste que desejar. Cada teste grava seus próprios CSVs em `../resultados/Txx_.../` (a partir da raiz `frtb`). Nenhuma célula de teste depende da execução de outra.

**Governança:** confirmar com a área data-base, população, controles de totalização, regras de classificação e se a extração contém todas as posições do conglomerado. Não subir dados ou saídas no GitHub público.

## Inicialização — caminhos e módulos
**Tabelas:** nenhuma. Importa funções, sem carregar os arquivos Excel.

In [ ]:
from pathlib import Path
from src import testes
from src.configuracao import ROOT, DADOS, RESULTADOS, ARQUIVOS
print("Raiz:", ROOT)
print("Arquivos esperados:", ARQUIVOS)

## Pré-checagem opcional — arquivos disponíveis
**Tabelas:** todas as bases configuradas. Não é necessário executar antes dos testes; útil para identificar nomes ausentes.

In [ ]:
import pandas as pd
from src.carregamento import caminho
pd.DataFrame([{"base":k,"arquivo":str(caminho(k)),"existe":caminho(k).exists()} for k in ARQUIVOS])

## T01 — Integridade da população

**Tabelas:** `Base_Posicoes_Completa`.

**Objetivo:** Identifica chaves ausentes e repetidas. Chaves repetidas são alertas, não necessariamente erros.

**Evidências:** arquivos CSV e `resumo.csv` na pasta específica do teste. **Interpretação:** um status sem divergência não substitui validação da fonte, perímetro ou metodologia.

In [ ]:
resultado_t01_integridade = testes.t01_integridade()
pd.DataFrame([resultado_t01_integridade])

## T02 — Replicação independente do filtro SAS

**Tabelas:** `Base_Posicoes_Completa, IDs_Exposicao_Sens, Base_Posicoes`.

**Objetivo:** Reproduz a seleção via hash por (IDPos, CodCompraVenda) e compara o multiconjunto de registros, inclusive multiplicidades.

**Evidências:** arquivos CSV e `resumo.csv` na pasta específica do teste. **Interpretação:** um status sem divergência não substitui validação da fonte, perímetro ou metodologia.

In [ ]:
resultado_t02_replicacao_filtro = testes.t02_replicacao_filtro()
pd.DataFrame([resultado_t02_replicacao_filtro])

## T03 — Elegibilidade e rastreabilidade

**Tabelas:** `Base_Posicoes_Completa, IDs_Exposicao_Sens, Base_Posicoes`.

**Objetivo:** Identifica posições selecionadas sem ID elegível e sem registro na origem. Não prova elegibilidade regulatória sem regras externas.

**Evidências:** arquivos CSV e `resumo.csv` na pasta específica do teste. **Interpretação:** um status sem divergência não substitui validação da fonte, perímetro ou metodologia.

In [ ]:
resultado_t03_elegibilidade = testes.t03_elegibilidade()
pd.DataFrame([resultado_t03_elegibilidade])

## T04 — Duplicidades

**Tabelas:** `Base_Posicoes_Completa, IDs_Exposicao_Sens, Base_Posicoes`.

**Objetivo:** Distingue repetição de chave de duplicidade integral; exige análise de granularidade e compra/venda.

**Evidências:** arquivos CSV e `resumo.csv` na pasta específica do teste. **Interpretação:** um status sem divergência não substitui validação da fonte, perímetro ou metodologia.

In [ ]:
resultado_t04_duplicidades = testes.t04_duplicidades()
pd.DataFrame([resultado_t04_duplicidades])

## T05 — Classificação e ambiguidades

**Tabelas:** `IDs_Exposicao_Sens`.

**Objetivo:** Analisa distribuição de classes, ausências e diferentes classes associadas à mesma chave do hash.

**Evidências:** arquivos CSV e `resumo.csv` na pasta específica do teste. **Interpretação:** um status sem divergência não substitui validação da fonte, perímetro ou metodologia.

In [ ]:
resultado_t05_classificacao = testes.t05_classificacao()
pd.DataFrame([resultado_t05_classificacao])

## T06 — Integridade e vínculo dos fluxos

**Tabelas:** `Fluxos_Por_Cenario, Base_Posicoes`.

**Objetivo:** Procura fluxos sem posição correspondente e datas inconsistentes quando disponíveis. Reconciliação de valores exige regras contratuais.

**Evidências:** arquivos CSV e `resumo.csv` na pasta específica do teste. **Interpretação:** um status sem divergência não substitui validação da fonte, perímetro ou metodologia.

In [ ]:
resultado_t06_fluxos = testes.t06_fluxos()
pd.DataFrame([resultado_t06_fluxos])

## Interpretação e limitações

- **APROVADO:** nenhuma diferença encontrada nos critérios implementados, condicionado à integridade e comparabilidade das bases.
- **DIVERGENCIA:** divergência quantitativa que requer investigação e documentação.
- **ALERTA:** inconsistência potencial; não implica automaticamente falha do modelo.
- **CONCLUIDO_COM_RESSALVAS:** teste executado, mas não suficiente para atestar adequação metodológica ou regulatória.

### Próximas extensões
1. Reconciliação de população do conglomerado com controles da origem e com o perímetro prudencial.
2. Revisão de `CTR_Fatores` e do mapeamento de `CodFator`/`CodClasse` (dependente de extração e dicionário).
3. Reconciliação financeira de fluxos e preços, respeitando moeda, sinal, taxa e granularidade.
4. Replicação independente de delta, vega, curvatura e agregação SBM, quando disponíveis resultados, choques e parâmetros.

**Perguntas de validação:** o ID da posição é único por data-base? O filtro considera a mesma data-base de todas as tabelas? Como são tratadas posições sem fator mapeado? Há reconciliação com a contabilidade?